In [1]:
# 설치 끝나면 세션 다시 시작
!pip install -U transformers datasets==2.21.0 scipy scikit-learn

# 데이터 로딩

In [ ]:
from datasets import load_dataset

task = "ynat"
datasets = load_dataset("klue", task)

# https://huggingface.co/datasets/klue/klue

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/45678 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/9107 [00:00<?, ? examples/s]

In [ ]:
datasets

DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 9107
    })
})

In [ ]:
# 훈련 데이터 세트 확인
datasets['train'][0]

{'guid': 'ynat-v1_train_00000',
 'title': '유튜브 내달 2일까지 크리에이터 지원 공간 운영',
 'label': 3,
 'url': 'https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=105&sid2=227&oid=001&aid=0008508947',
 'date': '2016.06.30. 오전 10:36'}

In [ ]:
import pandas as pd
from IPython.display import display, HTML
from datasets import ClassLabel
import random
import numpy as np
def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."

    picks = []

    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)

        # 이미 등록된 예제가 뽑힌 경우, 다시 추출
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)

        picks.append(pick)

    # 임의로 추출된 인덱스들로 구성된 데이터 프레임 선언
    df = pd.DataFrame(dataset[picks])

    for column, typ in dataset.features.items():
        # 라벨 클래스를 스트링으로 변환
        if isinstance(typ, ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])

    display(HTML(df.to_html()))

In [ ]:
show_random_elements(datasets["train"])

,guid,title,label,url,date
0,ynat-v1_train_27059,중국 미국에 맞서 첨단무기 개발 박차,세계,https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=104&sid2=231&oid=001&aid=0010697118,2019.03.15. 오후 12:48
1,ynat-v1_train_26298,학원비도 카톡으로…카카오페이 결제시스템 출시,IT과학,https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=105&sid2=226&oid=001&aid=0009486783,2017.08.21. 오전 10:41
2,ynat-v1_train_31834,美 북핵합의 주역 지금 北과 대화하면 핵보유국 인정 초래,세계,https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=100&sid2=268&oid=001&aid=0008721172,2016.10.01. 오전 8:37
3,ynat-v1_train_11111,제조업 취업자 11만5천명↓…감소규모 7년1개월 만에 최대2보,사회,https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=101&sid2=261&oid=001&aid=0008810070,2016.11.09. 오전 8:25
4,ynat-v1_train_45227,판문점 선언 환송공연 감상하는 남북정상,정치,https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=100&sid2=268&oid=001&aid=0010053964,2018.04.27. 오후 10:52
5,ynat-v1_train_40551,한파 속 산에서 길 잃은 50대 소방·경찰 공조로 구조,사회,https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=102&sid2=257&oid=001&aid=0009813039,2018.01.15. 오전 11:34
6,ynat-v1_train_06223,39개 주요 공공기관 부채비율 2022년 156%로 낮춘다,경제,https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=101&sid2=258&oid=001&aid=0010311937,2018.08.31. 오전 10:53
7,ynat-v1_train_17526,LG전자 100kW급 태양광 발전용 올인원 ESS 출시,경제,https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=101&sid2=263&oid=001&aid=0010523445,2018.12.13. 오전 10:00
8,ynat-v1_train_39533,코스피 삼성그룹주 강세에 상승 출발,경제,https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=101&sid2=258&oid=001&aid=0008731471,2016.10.06. 오전 9:25
9,ynat-v1_train_39208,이란 반경 400㎞ 드론·미사일 탐지 레이더 공개,세계,https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=104&sid2=234&oid=001&aid=0011012541,2019.08.10. 오후 7:24


**KLUE TC**

- 0 (IT과학)
- 1 (경제)
- 2 (사회)
- 3 (생활문화)
- 4 (세계)
- 5 (스포츠)
- 6 (정치)

# 평가(metric) 기준 설정


In [ ]:
# dataset 버전 3 부터는 없어짐 ㅠㅠ
from datasets import load_metric

metric = load_metric('f1')

<ipython-input-7-c3f9f9886600>:4: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric = load_metric('f1')


The repository for f1 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/f1.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


In [ ]:
fake_preds = np.random.randint(0, 2, size=(64,))
fake_labels = np.random.randint(0, 2, size=(64,))
fake_preds, fake_labels

(array([0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1,
        1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1,
        1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0]),
 array([0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0,
        1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0,
        1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1]))

In [ ]:
metric.compute(
    predictions=fake_preds, # 예측 값
    references=fake_labels # 타겟
)

{'f1': 0.5538461538461539}

# Tokenizer 설정

In [ ]:
# 한국어 NLU를 위한 대표적인 모델입니다. klue 말고도 많은 모델이 있습니다.
model_name = "klue/roberta-base"

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [ ]:
query = "프리미어 리그에서 활약할 손흥민 선수의 마지막 한국 인터뷰"

In [ ]:
tokenizer(query)

{'input_ids': [0, 12416, 4469, 27135, 5943, 2085, 11251, 3825, 2079, 4178, 3629, 5111, 2], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokenizer.cls_token_id

0

In [ ]:
tokenizer

BertTokenizerFast(name_or_path='klue/roberta-base', vocab_size=32000, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[CLS]', 'eos_token': '[SEP]', 'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [ ]:
# 데이터 전처리 함수 정의
def preprocess_function(data):
    return tokenizer(
        data['title'], # 데이터
        truncation=True, # 문장이 모델이 받아들일 수 있는 최대 길이 이상으로 들어올 경우 최대 길이를 기준으로 시퀀스(문장) 자르기
        return_token_type_ids=False, # token_type_ids가 필요없는 RoBERTa 모델은 False로 설정
    )

In [ ]:
preprocess_function(datasets["train"][:3])

{'input_ids': [[0, 10637, 8474, 22, 2210, 2299, 2118, 28940, 3691, 4101, 3792, 2], [0, 24905, 1042, 4795, 19982, 2129, 121, 6904, 16311, 3, 14392, 2], [0, 4172, 3797, 3728, 2107, 2134, 3777, 904, 6022, 2332, 2113, 2259, 4523, 1380, 2259, 2062, 2]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

In [ ]:
# 모든 데이터에 대해 전처리 적용
encoded_datasets = datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/45678 [00:00<?, ? examples/s]

Map:   0%|          | 0/9107 [00:00<?, ? examples/s]

# Model 설정

In [ ]:
from transformers import AutoModelForSequenceClassification

num_labels = 7

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at klue/roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# 모델이 예측 했을 때 지표 계산
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    return metric.compute(predictions=predictions, references=labels, average = 'macro')

## Trainer 정의
이제 앞서 정의한 정보들을 바탕으로 `transformers`에서 제공하는 *Trainer* 객체를 활용하기 위한 인자 관리 클래스를 초기화합니다.

`metric_name`은 앞서 얻어진 메트릭 함수를 활용했을 때, 아래와 같이 `dict` 형식으로 결과 값이 반환되는데 여기서 우리가 사용할 *key* 를 정의해준다고 생각하시면 됩니다.

```python
>>> metric.compute(predictions=fake_preds, references=fake_labels)
{'f1': 0.85}
```

각 인자에 대한 자세한 설명은 [문서](https://huggingface.co/transformers/main_classes/trainer.html#trainingarguments)에서 참조해주시면 됩니다.

In [22]:
from transformers import TrainingArguments # Trainer 클래스에서 사용할 훈련용 하이퍼파라미터 정의

metric_name = "f1"
batch_size = 32

args = TrainingArguments(
    "test-tc", # 모델 결과물을 저장할 디렉토리
    evaluation_strategy = "epoch", # 평가 전략: 언제마다 평가할지 결정 (에폭이 끝날 때마다 평가)
    save_strategy = "epoch", # 모델 저장은 언제마다 할건지 (에폭이 끝날 때마다 저장)
    learning_rate=2e-5, # 학습률
    per_device_train_batch_size=batch_size, # 분산 훈련 장치의 배치 사이즈
    per_device_eval_batch_size=batch_size, # 분산 평가 장치의 배치 사이즈

    num_train_epochs=3, # 학습 에폭 설정
    weight_decay=0.01, # 가중치 감쇠 상수

    load_best_model_at_end = True, # 가장 성능이 좋았던 모델을 훈련이 끝나고 자동으로 불러오는 옵션
    metric_for_best_model = metric_name,
    )

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [23]:
from transformers import Trainer # TrainingArguments를 입력 받아 훈련을 수행

trainer = Trainer(
    model, # 훈련 시킬 모델
    args, # TrainingArguments
    train_dataset = encoded_datasets["train"],
    eval_dataset = encoded_datasets["validation"],
    tokenizer = tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.392400,0.422296,0.860507
2,0.285600,0.369414,0.866993
3,0.212600,0.392570,0.868650


TrainOutput(global_step=4284, training_loss=0.31390618432024503, metrics={'train_runtime': 750.0464, 'train_samples_per_second': 182.701, 'train_steps_per_second': 5.712, 'total_flos': 1533172750755300.0, 'train_loss': 0.31390618432024503, 'epoch': 3.0})

# Evaluate

In [24]:
# 트레이너는 학습을 마치게 됐을 때 메트릭 기준 가장 좋은 성능을 보였던 모델 체크포인트를 로딩
trainer.evaluate()

{'eval_loss': 0.3925704061985016,
 'eval_f1': 0.8686504978435377,
 'eval_runtime': 12.1881,
 'eval_samples_per_second': 747.204,
 'eval_steps_per_second': 23.383,
 'epoch': 3.0}

# 파이프라인 생성

In [25]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="/content/test-tc/checkpoint-2856",
    return_all_scores=True
)

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
/usr/local/lib/python3.10/dist-packages/transformers/pipelines/text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [ ]:
print(query)

프리미어 리그에서 활약할 손흥민 선수의 마지막 한국 인터뷰


In [ ]:
classifier("기안 84가 갠지스 강물을 마셨습니다.")

[[{'label': 'LABEL_0', 'score': 0.0022082370705902576},
  {'label': 'LABEL_1', 'score': 0.001594709581695497},
  {'label': 'LABEL_2', 'score': 0.03219762071967125},
  {'label': 'LABEL_3', 'score': 0.6213455200195312},
  {'label': 'LABEL_4', 'score': 0.33913132548332214},
  {'label': 'LABEL_5', 'score': 0.0018390640616416931},
  {'label': 'LABEL_6', 'score': 0.0016835571732372046}]]

**KLUE TC**

- 0 (IT과학)
- 1 (경제)
- 2 (사회)
- 3 (생활문화)
- 4 (세계)
- 5 (스포츠)
- 6 (정치)